In [1]:
import air_quality

In [2]:
from pathlib import Path

In [3]:
import csv

In [4]:
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "data" / "openaq_varna_8843.csv"

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    print(reader.fieldnames)

['location_id', 'location_name', 'parameter', 'value', 'unit', 'datetimeUtc', 'datetimeLocal', 'timezone', 'latitude', 'longitude', 'country_iso', 'isMobile', 'isMonitor', 'owner_name', 'provider']


In [5]:
print(*reader.fieldnames, sep="\n")

location_id
location_name
parameter
value
unit
datetimeUtc
datetimeLocal
timezone
latitude
longitude
country_iso
isMobile
isMonitor
owner_name
provider


In [6]:
print(dir(reader.fieldnames))

['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [7]:
sep="\n"

In [8]:
name = "Varna"

In [9]:
print(name)

Varna


In [10]:
datetime_text = "2026-06-01T00:00:00+03:00"

In [11]:
date_part, time_part = datetime_text.split("T")

In [12]:
print(datetime_text)

2026-06-01T00:00:00+03:00


In [13]:
import air_quality 
air_quality.format_datetime(datetime_text)

'01.06.2026 00:00'

In [14]:
year, month, day = date_part.split("-")

In [15]:
print (date_part)

2026-06-01


In [16]:
date_part = "2026-06-01"
year, month, day = date_part.split("-")
print(year)
print(month)
print(day)

2026
06
01


In [17]:
formatted_date = f"{day}.{month}.{year}"

In [18]:
print(formatted_date)

01.06.2026


In [19]:
formatted_time = time_part[:5]

In [20]:
print(formatted_time)

00:00


In [21]:
empty_result = air_quality.format_datetime("")

In [22]:
print(empty_result == "")

True


In [23]:
parameter_counts = {}

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        parameter = row.get("parameter", "").strip()

        if parameter:
            parameter_counts[parameter] = parameter_counts.get(parameter, 0) + 1

print(parameter_counts)

{'co': 600, 'no2': 600, 'o3': 600, 'pm10': 600, 'pm25': 600, 'so2': 600}


In [24]:
value_text = row.get("value", "")

print(value_text)
print(type(value_text))

11.38
<class 'str'>


In [25]:
print("Parameter:", row.get("parameter", ""))
print("Value:", row.get("value", ""))
print("Unit:", row.get("unit", ""))
print("UTC:", row.get("datetimeUtc", ""))
print("Local:", row.get("datetimeLocal", ""))

Parameter: so2
Value: 11.38
Unit: µg/m³
UTC: 2026-07-02T00:00:00Z
Local: 2026-07-02T03:00:00+03:00


In [26]:
latest_rows = {}

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        parameter = row.get("parameter", "").strip()
        datetime_utc = row.get("datetimeUtc", "").strip()

        if not parameter or not datetime_utc:
            continue

        if parameter not in latest_rows:
            latest_rows[parameter] = row

        elif datetime_utc > latest_rows[parameter]["datetimeUtc"]:
            latest_rows[parameter] = row

In [27]:
print(sorted(latest_rows))

['co', 'no2', 'o3', 'pm10', 'pm25', 'so2']


In [28]:
for parameter in sorted(latest_rows):
    latest_row = latest_rows[parameter]

    print(
        f"{parameter}: "
        f"{latest_row['value']} {latest_row['unit']} | "
        f"UTC: {latest_row['datetimeUtc']} | "
        f"Local: {latest_row['datetimeLocal']}"
    )

co: 280 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
no2: 41 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
o3: 62.91 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
pm10: 24.41 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
pm25: 10.61 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
so2: 11.38 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00


In [29]:
blank_value_count = 0
non_numeric_value_count = 0
negative_value_count = 0
negative_examples = []

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        value_text = row.get("value", "").strip()

        if not value_text:
            blank_value_count += 1
            continue

        try:
            value = float(value_text)
        except ValueError:
            non_numeric_value_count += 1
            continue

        if value < 0:
            negative_value_count += 1

            if len(negative_examples) < 5:
                negative_examples.append(
                    {
                        "parameter": row.get("parameter", ""),
                        "value": value,
                        "datetimeUtc": row.get("datetimeUtc", ""),
                    }
                )

print("Blank values:", blank_value_count)
print("Non-numeric values:", non_numeric_value_count)
print("Negative values:", negative_value_count)
print("Negative examples:", negative_examples)

Blank values: 0
Non-numeric values: 0
Negative values: 142
Negative examples: [{'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-03T10:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T20:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T21:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T22:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T23:00:00Z'}]


In [30]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAQ_API_KEY")

print(api_key is not None)

True


In [32]:

import requests


url = "https://api.openaq.org/v3/locations/8843/latest"

headers = {
    "X-API-Key": api_key
}


response = requests.get(url, headers=headers, timeout=30)

print(response.status_code)

200


In [33]:

api_data = response.json()


print(type(api_data))

print(api_data.keys())


results = api_data.get("results", [])


print("Number of results:", len(results))

<class 'dict'>
dict_keys(['meta', 'results'])
Number of results: 7


In [34]:
print(api_data.keys())

dict_keys(['meta', 'results'])


In [35]:
results = api_data.get("results", [])

In [37]:
print("Number of results:", len(results))

Number of results: 7


In [38]:
if results:
    print(results[0])

{'datetime': {'utc': '2026-07-28T04:00:00Z', 'local': '2026-07-28T07:00:00+03:00'}, 'value': 75.53, 'coordinates': {'latitude': 43.22438900023895, 'longitude': 27.915733000170828}, 'sensorsId': 25777, 'locationsId': 8843}


In [39]:
print(results[0])

{'datetime': {'utc': '2026-07-28T04:00:00Z', 'local': '2026-07-28T07:00:00+03:00'}, 'value': 75.53, 'coordinates': {'latitude': 43.22438900023895, 'longitude': 27.915733000170828}, 'sensorsId': 25777, 'locationsId': 8843}


In [40]:
from pprint import pprint


first_result = results[0]

pprint(first_result, sort_dicts=False)

{'datetime': {'utc': '2026-07-28T04:00:00Z',
              'local': '2026-07-28T07:00:00+03:00'},
 'value': 75.53,
 'coordinates': {'latitude': 43.22438900023895,
                 'longitude': 27.915733000170828},
 'sensorsId': 25777,
 'locationsId': 8843}


In [41]:

sensors_url = (
    "https://api.openaq.org/v3/locations/8843/sensors"
)

sensors_response = requests.get(
    sensors_url,
    headers=headers,
    timeout=30,
)

print(sensors_response.status_code)

200


In [42]:

sensors_data = sensors_response.json()

sensor_results = sensors_data.get("results", [])

print("Number of sensors:", len(sensor_results))

for sensor in sensor_results:
    parameter = sensor.get("parameter", {})

    print(
        f"Sensor ID: {sensor.get('id')} | "
        f"Parameter: {parameter.get('name')} | "
        f"Unit: {parameter.get('units')}"
    )

Number of sensors: 7
Sensor ID: 25777 | Parameter: o3 | Unit: µg/m³
Sensor ID: 25778 | Parameter: no2 | Unit: µg/m³
Sensor ID: 25776 | Parameter: pm10 | Unit: µg/m³
Sensor ID: 25779 | Parameter: co | Unit: µg/m³
Sensor ID: 25774 | Parameter: so2 | Unit: µg/m³
Sensor ID: 25775 | Parameter: pm25 | Unit: µg/m³
Sensor ID: 4272879 | Parameter: no | Unit: µg/m³


In [43]:
from datetime import datetime


TARGET_PARAMETERS = {
    "co",
    "no2",
    "o3",
    "pm10",
    "pm25",
    "so2",
}



# sensor ID → parameter name и unit.
sensor_map = {}

for sensor in sensor_results:
    parameter_data = sensor.get("parameter", {})

    sensor_id = sensor.get("id")
    parameter_name = parameter_data.get("name", "")
    unit = parameter_data.get("units", "")

    if sensor_id is None or not parameter_name:
        continue

    sensor_map[sensor_id] = {
        "parameter": parameter_name.strip().lower(),
        "unit": unit,
    }


def format_api_datetime(datetime_text):
    if not datetime_text:
        return ""

    try:
        parsed_datetime = datetime.fromisoformat(
            datetime_text.replace("Z", "+00:00")
        )

        return parsed_datetime.strftime("%d.%m.%Y %H:%M")

    except ValueError:
        return datetime_text


live_measurements = {}

for result in results:
    sensor_id = result.get("sensorsId")

    sensor_info = sensor_map.get(sensor_id, {})

    parameter = sensor_info.get("parameter", "")
    unit = sensor_info.get("unit", "")

    if parameter not in TARGET_PARAMETERS:
        continue

    value = result.get("value")

    try:
        value = float(value)

    except (TypeError, ValueError):
        continue

    if value < 0:
        continue

    datetime_data = result.get("datetime", {})

    datetime_local_raw = datetime_data.get("local", "")
    datetime_utc_raw = datetime_data.get("utc", "")

    live_measurements[parameter] = {
        "datetime_local": format_api_datetime(
            datetime_local_raw
        ),
        "datetime_utc": format_api_datetime(
            datetime_utc_raw
        ),
        "datetime_utc_raw": datetime_utc_raw,
        "value": value,
        "unit": unit,
        "location_name": "AMS SOU Angel Kanchev-Varna",
    }


print("Live parameters:", sorted(live_measurements))

for parameter in sorted(live_measurements):
    measurement = live_measurements[parameter]

    print(
        f"{parameter}: "
        f"{measurement['value']} "
        f"{measurement['unit']} | "
        f"{measurement['datetime_local']}"
    )

Live parameters: ['co', 'no2', 'o3', 'pm10', 'pm25', 'so2']
co: 140.0 µg/m³ | 28.07.2026 07:00
no2: 23.96 µg/m³ | 28.07.2026 07:00
o3: 75.53 µg/m³ | 28.07.2026 07:00
pm10: 30.41 µg/m³ | 28.07.2026 07:00
pm25: 12.2 µg/m³ | 28.07.2026 07:00
so2: 16.31 µg/m³ | 28.07.2026 07:00
